# Resume Matcher with a Free LLM API
### Use a large language model through an API, check its answers, and ship it as a web app

**What we build:** paste a resume and a job description; an LLM returns a **match score**, the **skills that match**, the **skills that are missing**, and **tips** to improve the resume.

**What you learn today**
1. How to call an LLM through an API (the same code works for OpenAI, OpenRouter, Groq and others)
2. How to write a prompt that returns **structured data** (JSON) instead of a paragraph
3. How to **check** an LLM: is it consistent, is it fair, does it invent things?
4. How to keep an **API key secret**, in the notebook and in a deployed app
5. How to deploy the app on Streamlit Cloud through GitHub

**No GPU, no model training, no cost.** The model runs on OpenRouter's servers; we only send text and get text back.

## 0. Get a free OpenRouter API key (5 minutes, no credit card)
1. Go to **openrouter.ai** and sign in with Google or GitHub.
2. Open **Keys** (top-right menu), click **Create API Key**, give it a name such as `class`, and copy the key. It starts with `sk-or-`.
3. **Treat the key like a password.** Never paste it into a notebook cell, a screenshot, or a GitHub file.

**Store it safely**
- **Google Colab:** click the 🔑 key icon in the left sidebar → **Add new secret** → name `OPENROUTER_API_KEY`, paste the key, switch on *Notebook access*.
- **Your laptop:** the next cells will ask you to paste it into a hidden box.

**The free plan's limits:** 20 requests per minute and **50 requests per day**. This notebook uses about 26, and it **saves every answer** so re-running a cell does not spend another request.

## 1. Setup

In [ ]:
# On Colab, openai is usually installed already. On a laptop: pip install openai pandas matplotlib seaborn
import hashlib
import json
import os
import re
import time

import matplotlib.pyplot as plt
import openai
import pandas as pd
import seaborn as sns
from openai import OpenAI

# ----- settings -----
BASE_URL = "https://openrouter.ai/api/v1"   # where the requests go
MODEL = "openrouter/free"                   # OpenRouter picks an available free model for each request
CACHE_FILE = "llm_cache.json"               # saved answers, so we do not spend requests twice

print("openai library version:", openai.__version__)

In [ ]:
# Load the API key WITHOUT writing it in the notebook
API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata          # Colab's secret store (the key icon)
        API_KEY = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("Paste your OpenRouter API key (it will not be shown): ")

print(f"Key loaded: {len(API_KEY)} characters (we never print the key itself)")
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

## 2. Your first API call
An API call sends a list of **messages** and gets one reply back. Each message has a **role**:
- `system`: instructions about *how* the assistant should behave
- `user`: what we ask
- `assistant`: what the model answered (used when we continue a conversation)

In [ ]:
CALLS_MADE = 0


def ask(messages, temperature=0.0):
    # Send messages to the LLM and return the full response object
    global CALLS_MADE
    CALLS_MADE += 1
    return client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)


response = ask([
    {"role": "system", "content": "You are a helpful career coach. Answer in one sentence."},
    {"role": "user", "content": "What is an ATS in recruitment?"},
])

print("Answer:     ", response.choices[0].message.content)
print("Model used: ", response.model)
if response.usage:   # some providers do not report token counts
    print("Tokens:     ", response.usage.prompt_tokens, "in,", response.usage.completion_tokens, "out")

**Concepts**
- **Tokens** are pieces of words (roughly ¾ of a word each). Paid APIs charge per token; free plans limit requests.
- **`response.model`** shows which model answered. With `openrouter/free` it can change from call to call.

### Temperature: how random should the answer be?
`temperature=0` gives the most likely answer (good for scoring). Higher values give more varied answers (good for creative writing).

In [ ]:
question = [{"role": "user", "content": "Write a catchy one-line tagline for a resume-matching app."}]
for i in range(2):
    reply = ask(question, temperature=1.0).choices[0].message.content
    print(f"Try {i + 1}: {reply.strip()}")

## 3. Our sample data
`samples.json` holds **5 resumes** and **3 job descriptions**. Every person, company, email and phone number in it is **invented**.
Three of the resumes were written for one of the jobs each (`best_for`). That gives us a simple way to check the LLM later: *does the right resume get the top score?*

In [ ]:
if not os.path.exists("samples.json"):
    try:
        from google.colab import files
        print("Please upload samples.json from the class kit")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put samples.json next to this notebook")

with open("samples.json", encoding="utf-8") as f:
    samples = json.load(f)
resumes, jobs = samples["resumes"], samples["jobs"]

overview = pd.DataFrame([
    {"id": k, "type": "resume", "label": v["label"], "words": len(v["text"].split()), "written for": v["best_for"] or "-"}
    for k, v in resumes.items()
] + [
    {"id": k, "type": "job", "label": v["label"], "words": len(v["text"].split()), "written for": "-"}
    for k, v in jobs.items()
])
overview

In [ ]:
print(resumes["R1_data_analyst"]["text"][:600])

### A baseline without AI: keyword matching
Before using an LLM, try the simplest idea: count how many skills from the job description also appear in the resume.
Always build a simple baseline first; it tells you whether the fancy tool adds anything.

In [ ]:
SKILLS = ["sql", "excel", "power bi", "tableau", "python", "pandas", "scikit-learn", "pytorch", "docker",
          "fastapi", "flask", "aws", "mlflow", "react", "typescript", "javascript", "html", "css",
          "tailwind", "jest", "rest api", "figma", "jira", "seo", "google ads", "git"]


def find_skills(text):
    text = text.lower()
    return {skill for skill in SKILLS if skill in text}


def keyword_score(resume_text, job_text):
    job_skills = find_skills(job_text)
    if not job_skills:
        return 0
    return round(100 * len(job_skills & find_skills(resume_text)) / len(job_skills))


keyword_table = pd.DataFrame(
    {r: {j: keyword_score(resumes[r]["text"], jobs[j]["text"]) for j in jobs} for r in resumes})
keyword_table

Keyword matching is fast and free, but it is blind: it misses synonyms ("deep learning framework" vs "PyTorch"), ignores years of experience, and cannot explain anything. Let's see what an LLM adds.

## 4. Prompt design: from a vague question to structured data
### Version 1: a vague prompt

In [ ]:
resume_text = resumes["R5_business_analyst"]["text"]
job_text = jobs["J1_data_analyst"]["text"]

vague = ask([{"role": "user", "content": f"Does this resume match this job?\n\nJOB:\n{job_text}\n\nRESUME:\n{resume_text}"}])
print(vague.choices[0].message.content)

A good answer for a human, but useless for an app: where is the score? Which skills are missing? The format changes every time.

### Version 2: a structured prompt
A good prompt has four parts:
1. **Role**: who the model should act as (system message)
2. **Task**: exactly what to do
3. **Rubric and rules**: how to decide, so scores are consistent
4. **Output format**: the exact JSON shape we want back

In [ ]:
SYSTEM_PROMPT = ("You are an experienced technical recruiter. You judge resumes fairly and only use facts "
                 "written in the resume. You never consider a candidate's name, gender, age or location.")


def build_prompt(resume_text, job_text):
    return f'''Compare the RESUME with the JOB DESCRIPTION.

Score the match from 0 to 100 using this rubric:
- Must-have skills in the job: up to 50 points
- Relevant experience (years and type of work): up to 25 points
- Nice-to-have skills: up to 15 points
- Education and projects: up to 10 points

Rules:
- Only count a skill as matched if the resume clearly shows it.
- Keep each list item short (2-6 words).

Reply with JSON only, no other text, in exactly this format:
{{
  "match_score": <number 0-100>,
  "verdict": "<Strong match | Partial match | Weak match>",
  "matched_skills": ["..."],
  "missing_skills": ["..."],
  "strengths": ["...", "..."],
  "suggestions": ["...", "...", "..."],
  "summary": "<one or two sentences>"
}}

JOB DESCRIPTION:
<job>
{job_text}
</job>

RESUME:
<resume>
{resume_text}
</resume>'''


structured = ask([{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": build_prompt(resume_text, job_text)}])
raw_reply = structured.choices[0].message.content
print(raw_reply)

*(The double braces `{{ }}` are how you write a literal `{ }` inside a Python f-string.)*

## 5. Turn the reply into Python data, safely
LLMs do not always follow instructions perfectly. Some wrap the JSON in ```` ```json ```` fences, some add a sentence before it, some return a score as text.
**Never trust the format; check it.** These two small functions make the reply safe to use.

In [ ]:
def extract_json(reply):
    # Find the {...} part of the reply and turn it into a Python dict
    reply = re.sub(r"```(?:json)?", "", reply or "")
    start, end = reply.find("{"), reply.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON found in the reply")
    return json.loads(reply[start:end + 1])


def clean_result(data):
    # Make sure every field exists and has the right type
    def as_list(value):
        if isinstance(value, list):
            return [str(v).strip() for v in value if str(v).strip()]
        return [str(value).strip()] if value else []

    try:
        score = int(round(float(data.get("match_score", 0))))
    except (TypeError, ValueError):
        score = 0
    return {
        "match_score": max(0, min(100, score)),
        "verdict": str(data.get("verdict", "")).strip(),
        "matched_skills": as_list(data.get("matched_skills")),
        "missing_skills": as_list(data.get("missing_skills")),
        "strengths": as_list(data.get("strengths")),
        "suggestions": as_list(data.get("suggestions")),
        "summary": str(data.get("summary", "")).strip(),
    }


try:
    result = clean_result(extract_json(raw_reply))
    print(result)
except (ValueError, json.JSONDecodeError) as error:
    print("This reply had no valid JSON:", error)
    print("It happens with real LLMs! Section 6 adds an automatic retry for exactly this case.")

In [ ]:
# Test the parser on the kinds of messy replies LLMs really give
messy_replies = [
    '```json\n{"match_score": "72", "verdict": "Partial match"}\n```',
    'Sure! Here is the result: {"match_score": 140, "matched_skills": "SQL"} Hope this helps.',
]
for reply in messy_replies:
    print(clean_result(extract_json(reply)))

## 6. Privacy first: remove contact details before sending
Free models may log prompts, and some free providers use them to improve their models. A resume is personal data.
So before anything leaves our computer we **remove emails and phone numbers**. The LLM does not need them to judge skills.

In [ ]:
def redact_contacts(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "[email removed]", text)
    text = re.sub(r"\+?\d(?:[\s-]?\d){9,12}", "[phone removed]", text)
    return text


print(redact_contacts(resumes["R1_data_analyst"]["text"])[:120])

### Put it all together in one function
`match_resume` redacts, builds the prompt, calls the API, parses the reply (asking once more if the JSON is broken), and **saves the answer** so the same question is never paid for twice.
This is exactly the function the web app will use.

In [ ]:
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}


def match_resume(resume_text, job_text, use_cache=True):
    resume_text = redact_contacts(resume_text)
    prompt = build_prompt(resume_text, job_text)
    # The saved-answer key includes the model and the full prompt, so editing the prompt means new answers
    key = hashlib.md5(f"{MODEL}|{SYSTEM_PROMPT}|{prompt}".encode()).hexdigest()
    if use_cache and key in cache:
        return dict(cache[key], from_cache=True)

    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    for attempt in (1, 2):
        response = ask(messages)
        reply = response.choices[0].message.content or ""
        try:
            result = clean_result(extract_json(reply))
            break
        except (ValueError, json.JSONDecodeError):
            if attempt == 2:
                raise
            messages += [{"role": "assistant", "content": reply},
                         {"role": "user", "content": "That was not valid JSON. Reply again with the JSON only."}]

    result["model_used"] = response.model
    cache[key] = result
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=1)
    return dict(result, from_cache=False)


def show(result):
    print(f"Score: {result['match_score']}/100 - {result['verdict']}   (model: {result['model_used']})")
    print("Matched:", ", ".join(result["matched_skills"]))
    print("Missing:", ", ".join(result["missing_skills"]))
    print("Tips:   ", *result["suggestions"], sep="\n  - ")
    print("Summary:", result["summary"])


show(match_resume(resumes["R1_data_analyst"]["text"], jobs["J1_data_analyst"]["text"]))
print(f"\nAPI calls used so far: {CALLS_MADE} (free limit: 50 per day)")

## 7. Is the LLM any good? Three quick checks
An LLM always sounds confident. A data scientist checks it.

### Check 1: does the right resume win for each job?
Score all 5 resumes against all 3 jobs (15 calls, most of them new). We wait a few seconds between calls to stay under 20 requests per minute.

In [ ]:
llm_scores = {}
for job_id, job in jobs.items():
    for resume_id, resume in resumes.items():
        result = match_resume(resume["text"], job["text"])
        llm_scores[(job_id, resume_id)] = result["match_score"]
        print(f"{job_id:16s} x {resume_id:20s} -> {result['match_score']:3d}  ({'saved' if result['from_cache'] else result['model_used']})")
        if not result["from_cache"]:
            time.sleep(3.5)   # stay under 20 requests per minute

llm_table = pd.Series(llm_scores).unstack()
print(f"\nAPI calls used so far: {CALLS_MADE}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
sns.heatmap(keyword_table, annot=True, fmt="d", cmap="Blues", vmin=0, vmax=100, ax=axes[0], cbar=False)
axes[0].set_title("Keyword baseline (% of job skills found)")
sns.heatmap(llm_table.loc[list(jobs), list(resumes)], annot=True, fmt="d", cmap="Greens", vmin=0, vmax=100, ax=axes[1], cbar=False)
axes[1].set_title("LLM match score")
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

expected = {r["best_for"]: rid for rid, r in resumes.items() if r["best_for"]}
check = pd.DataFrame({
    "expected best resume": expected,
    "LLM's top resume": {j: llm_table.loc[j].idxmax() for j in jobs},
    "keyword top resume": {j: keyword_table.loc[j].idxmax() for j in jobs},
})
check["LLM correct?"] = check["expected best resume"] == check["LLM's top resume"]
check

**Discussion:** both methods should find the obvious best fit. Look instead at the **second-best** scores. The business analyst (R5) knows SQL, Excel and some Power BI: a fair *partial* match for the data analyst job. Does the LLM see that? Does the keyword baseline?

### Check 2: is it consistent?
Ask the **same question 3 times** (skipping the saved answers). A reliable scorer should give similar numbers.

In [ ]:
repeat = []
for i in range(3):
    r = match_resume(resumes["R5_business_analyst"]["text"], jobs["J1_data_analyst"]["text"], use_cache=False)
    repeat.append({"try": i + 1, "score": r["match_score"], "verdict": r["verdict"], "model": r["model_used"]})
    time.sleep(3.5)
pd.DataFrame(repeat)

Scores can move even at `temperature=0`, and `openrouter/free` may send each call to a **different model**. For a real product you would fix one model (set `MODEL` to a specific `...:free` model) and keep the rubric.

### Check 3: does it invent skills?
The prompt says "only count skills the resume clearly shows". Let's verify: is every matched skill actually written in the resume?

In [ ]:
STOPWORDS = {"and", "with", "the", "for", "of", "in", "experience", "skills", "strong", "basic", "using"}


def unsupported_skills(result, resume_text):
    # Matched skills that share no meaningful word with the resume: possible inventions
    text = resume_text.lower()
    flagged = []
    for skill in result["matched_skills"]:
        words = [w for w in re.findall(r"[a-z0-9+#.]+", skill.lower()) if len(w) > 2 and w not in STOPWORDS]
        if words and not any(w in text for w in words):
            flagged.append(skill)
    return flagged


for resume_id in resumes:
    r = match_resume(resumes[resume_id]["text"], jobs["J1_data_analyst"]["text"])   # saved answers, no new calls
    print(f"{resume_id:20s} possibly invented: {unsupported_skills(r, resumes[resume_id]['text']) or 'none'}")

### Bonus check: is it fair?
Same resume, only the **name** changed. A fair scorer gives the same result. This costs 2 calls.

In [ ]:
original = resumes["R5_business_analyst"]["text"]
renamed = original.replace("SARA KHAN", "SAMEER KHAN")
a = match_resume(original, jobs["J1_data_analyst"]["text"], use_cache=False)
b = match_resume(renamed, jobs["J1_data_analyst"]["text"], use_cache=False)
print(f"Sara:   {a['match_score']}  ({a['model_used']})")
print(f"Sameer: {b['match_score']}  ({b['model_used']})")
print(f"\nAPI calls used today in this notebook: {CALLS_MADE}")

One pair proves little: a difference could come from randomness or a different model answering. Real fairness testing uses hundreds of pairs.
This is why the system prompt says *never consider name, gender, age or location*, and why automated resume screening is regulated in several countries.

## 8. From notebook to app
The app (`app.py` in the kit) reuses **the same functions**: `redact_contacts`, `build_prompt`, `extract_json`, `clean_result`, and the API call. It adds:
- a text box *or* PDF upload for the resume
- buttons to load the sample resumes and jobs
- a score card with matched and missing skills
- the API key read from **Streamlit Secrets**, never from the code

**Run it on your laptop:**
```bash
cd resume-matcher-app
pip install -r requirements.txt
mkdir .streamlit
# create .streamlit/secrets.toml containing one line:  OPENROUTER_API_KEY = "sk-or-..."
streamlit run app.py
```
`.gitignore` already stops `secrets.toml` from being uploaded to GitHub.

## Recap
| Idea | What we saw |
|---|---|
| LLM API | Messages in (system, user), text out; `response.model` and token usage |
| Temperature | 0 for scoring, higher for creative text |
| Baseline first | Keyword matching is free and fast; the LLM must beat it to be worth it |
| Prompt design | Role, task, rubric, exact JSON format |
| Never trust the format | `extract_json` + `clean_result` |
| Checking an LLM | Right answer on known cases, consistency, invented skills, fairness |
| Privacy and cost | Redact personal data; cache answers; stay inside 20/min and 50/day |
| Secrets | Colab secrets / hidden input in notebooks; Streamlit Secrets in apps; never in GitHub |